# Ejemplo aplicado: clasificación multiclase — Iris

Misma plantilla que [03-clasificacion-multiple.ipynb](03-clasificacion-multiple.ipynb), configurada para `iris.csv` (target `species`, 3 clases).

> Para un CSV nuevo, parte de la **plantilla genérica** correspondiente, no de este archivo.


In [ ]:
# --- Imports ---
import importlib
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")


def infer_feature_columns(df, target_col, drop_cols, feature_cols):
    """Columnas que entran como X. Si FEATURE_COLS es None, todas salvo target y DROP_COLS."""
    if feature_cols is not None:
        return list(feature_cols)
    exclude = {target_col, *drop_cols}
    return [c for c in df.columns if c not in exclude]


def infer_column_types(X, numeric_cols=None, categorical_cols=None):
    """Separa numéricas y categóricas para ColumnTransformer."""
    if numeric_cols is None:
        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    if categorical_cols is None:
        categorical_cols = X.select_dtypes(
            include=["object", "category", "bool", "string"]
        ).columns.tolist()
    return list(numeric_cols), list(categorical_cols)


def build_preprocess(numeric_cols, categorical_cols):
    """Preprocesador compartido: imputar → escalar / one-hot. Fit solo en train (Pipeline)."""
    transformers = []
    if numeric_cols:
        transformers.append(
            (
                "num",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_cols,
            )
        )
    if categorical_cols:
        transformers.append(
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "encoder",
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        ),
                    ]
                ),
                categorical_cols,
            )
        )
    if not transformers:
        raise ValueError("No hay columnas numéricas ni categóricas para preprocesar.")
    return ColumnTransformer(transformers, remainder="drop")


def _optional_estimator(module_name, class_name, **kwargs):
    try:
        module = importlib.import_module(module_name)
        cls = getattr(module, class_name)
        return cls(**kwargs)
    except ImportError:
        print(f"  [aviso] {module_name}.{class_name} no instalado (pip install {module_name})")
        return None


## 1. Explorar el CSV (antes de CONFIG)

Mismos parámetros que usarás en CONFIG.

In [ ]:
# --- Ajusta solo estas dos líneas para TU csv ---
PREVIEW_PATH = "data/iris.csv"
PREVIEW_SEP = ','

df_preview = pd.read_csv(PREVIEW_PATH, sep=PREVIEW_SEP)

print(f"Filas: {len(df_preview):,}  |  Columnas: {len(df_preview.columns)}")
print("\n--- Nombres de columnas ---")
for i, col in enumerate(df_preview.columns):
    print(f"  {i:2d}: {col!r}")

print("\n--- Tipos (dtypes) ---")
print(df_preview.dtypes)

print("\n--- Primeras filas ---")
display(df_preview.head())

print("\n--- Valores faltantes ---")
missing = df_preview.isna().sum()
if missing.any():
    display(missing[missing > 0].to_frame("nulos"))
else:
    print("No hay valores faltantes.")

_num = df_preview.select_dtypes(include=[np.number]).columns.tolist()
_cat = df_preview.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()
print("\n--- Sugerencia de tipos ---")
print("Numéricas:", _num)
print("Categóricas:", _cat)

if "species" in df_preview.columns:
    print("\n--- Distribución del posible target ---")
    print(df_preview["species"].value_counts())

print(
    "\n>>> Siguiente: en CONFIG usa el mismo path/separador y define TARGET_COL "
    "(columna con **varias clases** distintas)."
)


## 2. CONFIG — adaptar a tu dataset

Copia los valores de la exploración. **Solo esta sección** cambia entre proyectos.


In [ ]:
# ========== CONFIG ==========
DATA_PATH = "data/iris.csv"
CSV_SEP = ","

TARGET_COL = "species"

DROP_COLS = []
FEATURE_COLS = None
NUMERIC_COLS = None
CATEGORICAL_COLS = None

TEST_SIZE = 0.2
RANDOM_STATE = 42
METRIC_PRINCIPAL = "accuracy"

def build_models():
    from sklearn.ensemble import (
        GradientBoostingClassifier,
        HistGradientBoostingClassifier,
        RandomForestClassifier,
    )
    from sklearn.linear_model import LogisticRegression

    models = {
        "LogisticRegression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "RandomForest": RandomForestClassifier(
            n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
        ),
        "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
    }
    xgb = _optional_estimator(
        "xgboost", "XGBClassifier",
        random_state=RANDOM_STATE, verbosity=0, n_estimators=100,
        eval_metric="logloss", n_jobs=-1,
    )
    if xgb is not None:
        models["XGBoost"] = xgb
    cat = _optional_estimator(
        "catboost", "CatBoostClassifier",
        random_state=RANDOM_STATE, verbose=False, iterations=100,
        allow_writing_files=False,
    )
    if cat is not None:
        models["CatBoost"] = cat
    return models


MODELS = build_models()


## 3. Carga de datos


In [ ]:
df = pd.read_csv(DATA_PATH, sep=CSV_SEP)
print("Shape:", df.shape)
df.head()

## 4. Calidad de datos


In [ ]:
print(df[TARGET_COL].value_counts())
print("\nFaltantes:")
print(df.isna().sum().pipe(lambda s: s[s > 0] if s.any() else "Sin faltantes"))

## 5. Visualización rápida


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET_COL].value_counts().plot(kind="bar", ax=ax)
ax.set_title("Distribución de clases")
ax.set_xlabel(TARGET_COL)
plt.tight_layout()
plt.show()

## 6. X / y y split (estratificado)


In [ ]:
feature_cols = infer_feature_columns(df, TARGET_COL, DROP_COLS, FEATURE_COLS)
X = df[feature_cols]
y = df[TARGET_COL]
numeric_cols, categorical_cols = infer_column_types(X, NUMERIC_COLS, CATEGORICAL_COLS)
print("Numéricas:", len(numeric_cols), "| Categóricas:", len(categorical_cols))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

## 7. Preprocesado


In [ ]:
preprocess = build_preprocess(numeric_cols, categorical_cols)

## 8. Comparar modelos


In [ ]:
def classification_metrics(y_true, y_pred):
    from sklearn.metrics import (
        accuracy_score,
        f1_score,
        precision_score,
        recall_score,
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


def evaluate_models(models, preprocess, X_train, X_test, y_train, y_test):
    rows = []
    for name, estimator in models.items():
        pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        rows.append({"modelo": name, **classification_metrics(y_test, y_pred)})
    return pd.DataFrame(rows).sort_values(METRIC_PRINCIPAL, ascending=False)


results = evaluate_models(MODELS, preprocess, X_train, X_test, y_train, y_test)
display(results.round(4))

ax = results.plot(x="modelo", y=METRIC_PRINCIPAL, kind="barh", legend=False, figsize=(8, 5))
ax.set_xlabel(f"{METRIC_PRINCIPAL} (test)")
ax.set_title("Comparación de modelos — clasificación multiclase")
plt.tight_layout()
plt.show()


## 9. Detalle del mejor modelo


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

best_name = results.iloc[0]["modelo"]
print(f"Mejor modelo (test): {best_name}\n")

best_pipe = Pipeline([("preprocess", preprocess), ("model", MODELS[best_name])])
best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_test)

print(classification_report(y_test, y_pred))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title(f"Matriz de confusión — {best_name}")
plt.tight_layout()
plt.show()


## Checklist: nuevo dataset (multiclase)

1. CSV en `data/` → explorar → CONFIG.
2. `TARGET_COL` con **3 o más** clases distintas.
3. `DROP_COLS` e ids; revisa faltantes y tipos.
4. Split **estratificado** por `y`.
5. Métrica principal habitual: **accuracy** o **F1 weighted**.
